# Status Update 1, SMC FVG Auto-Detection on SPY

**Abstract.** A deep learning system to automatically detect Smart Money Concept (SMC) Fair Value Gap (FVG) zones from SPY hourly candlestick data. Three models trained and rigorously evaluated through May 13, 2026: XGBoost, LSTM, and CNN-LSTM. XGBoost leads at macro F1 = 0.721 ± 0.001 across 5 seeds. Both DL models trail but the full experimental infrastructure is in place. The next phase covers xLSTM, Transformer, and ensemble experiments.

*Nikola Baburov, Individual Research Project, Fontys ICT, Semester 6, May 17, 2026*

---

## 1. What we're building

One sentence: automatically detect Fair Value Gap zones on an hourly SPY chart, replacing a manual and subjective trader process with a trained classifier.

SMC (Smart Money Concepts) is a trading methodology that retail traders use to read price action as institutional participants are assumed to act. One of its core structures is the Fair Value Gap (FVG): a 3-candle pattern where a fast move leaves a price imbalance, creating a "gap" zone the market tends to revisit.

Traders find these zones by hand, which creates three problems. It's slow, every trade setup requires manual chart scanning. It's subjective, two traders looking at the same chart will disagree on which FVGs are "real." And it's hard to backtest: without systematic labels, there's no way to evaluate quality at scale.

This project trains a model to label each hourly bar as no FVG, bullish FVG, or bearish FVG, with confidence scores. The output is a Plotly chart overlay showing predicted zones over real SPY price history.

This is not price prediction. The model answers: "is this bar the confirmation point of a high-quality FVG?", not "will price go up?"

### What is an FVG?

Think of a queue at a ticket counter. Under normal conditions everyone gets served in turn and no gap forms. When a very large group pushes through all at once, they skip several spots and leave a gap the queue tries to fill later. An FVG is the same idea: a strong impulse candle moves so fast it skips a price range entirely, leaving an area where no normal two-sided trading occurred. That skipped range is the gap zone, and SMC traders expect price to return to it.

The 3-candle structure is:

- **Candle N-1**, the setup bar: its **high** (bullish) or **low** (bearish) defines one edge of the gap
- **Candle N**, the impulse candle: a strong directional move
- **Candle N+1**, the confirmation candle: its range does not overlap candle N-1's range

Bullish FVG condition: `high[N-1] < low[N+1]`, the gap zone is [N-1 high, N+1 low].

Bearish FVG condition: `low[N-1] > high[N+1]`, the gap zone is [N+1 high, N-1 low].

Geometrically the pattern is complete at N+1. That is the raw (basic) FVG, and Figure 5 shows it below.

#### What this project actually trains on, ValidFVG

The raw geometric rule fires on roughly 25% of all bars. Most are not meaningful trade setups. This project trains on a stricter label: the **ValidFVGLabeller**, which requires all six criteria to be met:

| # | Criterion | Description |
|---|-----------|-------------|
| 1 | Geometric gap | Basic 3-candle condition above |
| 2 | Reaction candle | Candle N+2 closes back into the gap zone |
| 3 | S/R confluence | A pivot high/low sits within the ATR-expanded zone (optional) |
| 4 | Priority labelling | All passing candidates are labelled; no elimination |
| 5 | Gann box position | FVG sits in the correct quadrant relative to the 50-bar swing midpoint |
| 6 | Break of Structure | A directional BOS occurred within the past 50 bars before candle N |

Because criterion 2 requires candle N+2 to have closed, the **ValidFVG label sits at N+2**, not N+1. Placing it at N+1 would require knowing N+2's close in advance, which is lookahead leakage. The ValidFVG label fires on roughly 3% of bars, an 8× reduction from the raw geometric rate.

Figure 5 below shows the basic 3-candle geometry. Section 3 (data) covers the full 6-criteria implementation.


In [1]:
"""
Figure 5 — Annotated FVG diagrams (bullish + bearish).
Synthetic OHLCV data — illustrates the 3-candle pattern.
"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Bullish FVG", "Bearish FVG"],
    horizontal_spacing=0.12,
)

# ------ BULLISH FVG ------
# N-1: down candle, N: big impulse up, N+1: gap confirmation, N+2: reaction
b_opens  = [100, 99, 104, 106]
b_highs  = [101, 100, 107, 107]
b_lows   = [98,  96, 103, 104]
b_closes = [99, 100, 106, 105]
x_labels = ["N-1", "N", "N+1", "N+2"]

for i, (o, h, l, c, lbl) in enumerate(zip(b_opens, b_highs, b_lows, b_closes, x_labels)):
    color = "#ef4444" if c < o else "#22c55e"
    fig.add_trace(go.Candlestick(
        x=[lbl], open=[o], high=[h], low=[l], close=[c],
        increasing_line_color="#22c55e", decreasing_line_color="#ef4444",
        increasing_fillcolor="#22c55e", decreasing_fillcolor="#ef4444",
        showlegend=False, name=lbl,
    ), row=1, col=1)

# Gap zone = N-1 high (101) to N+1 low (103)
fig.add_shape(
    type="rect", x0=-0.4, x1=3.4, y0=101, y1=103,
    fillcolor="rgba(34,197,94,0.2)", line_color="rgba(34,197,94,0.6)", line_width=1.5,
    row=1, col=1,
)
fig.add_annotation(x="N", y=102, text="Gap zone<br>[N-1 high → N+1 low]",
                   font=dict(size=11, color="#15803d"), showarrow=False,
                   bgcolor="rgba(255,255,255,0.8)", row=1, col=1)
fig.add_annotation(x="N-1", y=96.5, text="N-1<br>high=101", font=dict(size=10), showarrow=False, row=1, col=1)
fig.add_annotation(x="N+1", y=95.5, text="N+1<br>low=103", font=dict(size=10), showarrow=False, row=1, col=1)
fig.add_annotation(x="N+2", y=107.5, text="N+2\nreaction ↓", font=dict(size=10, color="#6b7280"), showarrow=False, row=1, col=1)

# ------ BEARISH FVG ------
# N-1: up candle, N: impulse down, N+1: gap confirmation, N+2: reaction up
r_opens  = [106, 107, 102, 100]
r_highs  = [108, 108, 104, 103]
r_lows   = [104, 101, 99,  99]
r_closes = [107, 103, 100, 101]

for i, (o, h, l, c, lbl) in enumerate(zip(r_opens, r_highs, r_lows, r_closes, x_labels)):
    fig.add_trace(go.Candlestick(
        x=[lbl], open=[o], high=[h], low=[l], close=[c],
        increasing_line_color="#22c55e", decreasing_line_color="#ef4444",
        increasing_fillcolor="#22c55e", decreasing_fillcolor="#ef4444",
        showlegend=False, name=lbl,
    ), row=1, col=2)

# Gap zone = N+1 high (104) to N-1 low (104) → use 104 to 108
fig.add_shape(
    type="rect", x0=-0.4, x1=3.4, y0=104, y1=108,
    fillcolor="rgba(239,68,68,0.2)", line_color="rgba(239,68,68,0.6)", line_width=1.5,
    row=1, col=2,
)
fig.add_annotation(x="N", y=106, text="Gap zone<br>[N+1 high → N-1 low]",
                   font=dict(size=11, color="#991b1b"), showarrow=False,
                   bgcolor="rgba(255,255,255,0.8)", row=1, col=2)
fig.add_annotation(x="N+2", y=98.5, text="N+2\nreaction ↑", font=dict(size=10, color="#6b7280"), showarrow=False, row=1, col=2)

fig.update_layout(
    title={
        "text": ("Figure 5. Bullish (left) and bearish (right) Fair Value Gap patterns.<br>"
                 "<sup>Synthetic OHLCV | Label placed at N+1 (first bar where pattern is fully confirmed)</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=430,
    font=dict(size=12),
    margin=dict(t=110, b=40),
    xaxis=dict(showgrid=False),
    xaxis2=dict(showgrid=False),
    yaxis=dict(showgrid=False),
    yaxis2=dict(showgrid=False),
)
# Hide range slider
for axis in ["xaxis", "xaxis2"]:
    fig.update_layout(**{f"{axis}_rangeslider_visible": False})

# fig.write_html("../reports/figure05_fvg_diagram.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure05_fvg_diagram.html")


*Figure 5. The bullish FVG (left) requires that candle N+1's low sits strictly above candle N-1's high, leaving a price gap. The shaded rectangle is the gap zone -- the imbalance SMC traders expect price to revisit. For the raw geometric pattern shown here, the label sits at N+1 (the first bar where all three candles are closed and the pattern is confirmed). The ValidFVGLabeller used as the actual training target places the label one bar later at N+2, because it requires candle N+2 to close back into the gap before confirming the setup.*

In [2]:
"""
Figure 1 — An FVG overlay on real SPY hourly data.

This cell loads an existing Plotly inspect report (generated 2026-05-13) and
displays it inline. The chart shows CNN-LSTM predictions on the 2023-2025 test slice.
Green triangles = bullish FVG predictions. Red triangles = bearish FVG predictions.
"""
import os
from IPython.display import IFrame, display, HTML

INSPECT_DIR = "../reports/inspect/2026-05-13_004532/plots"
TIMELINE_HTML = os.path.join(INSPECT_DIR, "timeline_lstm.html")

# Fall back to a window plot if timeline not present for cnn_lstm session
WINDOW_PLOT = os.path.join(INSPECT_DIR.replace("004532", "004532"), "window_001125.html")

# Use the inspect timeline from the prior session (lstm + xgb), which has the full overlay
PRIOR_INSPECT = "../reports/inspect/2026-05-12_223352/plots/timeline_lstm.html"

if os.path.exists(PRIOR_INSPECT):
    display(IFrame(src=PRIOR_INSPECT, width="100%", height=500))
    print("Figure 1. LSTM model predictions overlaid on SPY hourly chart (2023-2025 test slice).")
    print("Green markers = bullish FVG prediction, red = bearish, grey = no FVG.")
else:
    print(f"Inspect HTML not found at {PRIOR_INSPECT}")
    print("Run: python scripts/inspect_models.py --models lstm xgboost --lookahead-bars 20")

Figure 1. LSTM model predictions overlaid on SPY hourly chart (2023-2025 test slice).
Green markers = bullish FVG prediction, red = bearish, grey = no FVG.


*Figure 1. LSTM model predictions on the 2023-2025 SPY test slice. Each marker is a bar where the model predicted a bullish (green) or bearish (red) FVG zone. Marker density relative to the raw FVG rate (~3%) reflects the model's precision-recall tradeoff at argmax decoding.*

---

## 2. Why this is a deep learning problem

The naive FVG detector is pure geometry: find 3 consecutive candles where candle N-1's high is below candle N+1's low (bullish), or candle N-1's low is above candle N+1's high (bearish). That's 20 lines of pandas.

The problem: this geometric rule fires on roughly 25% of all hourly bars. Most of those aren't meaningful trade setups.

SMC traders filter the noise by applying additional judgment. A "valid" FVG typically requires a preceding Break of Structure showing directional intent, a price reaction at a support or resistance level, session context (New York open FVGs outweigh mid-session ones), a minimum gap size, no prior price mitigation (if the gap was already filled, it's inactive), and Gann or higher-timeframe confluence.

No single rule captures all of this. Experienced traders combine the criteria intuitively, but the thresholds are personal, vary across market regimes, and often get stated qualitatively ("the gap should be significant"). The ValidFVG labeller in this project implements those 6 criteria programmatically, producing a label set where only ~3% of bars qualify. That 3% set is the training target.

Why not just ship the 6-criteria rule as the "model"? Because hard-coding the thresholds is brittle. The values reflect one trader's interpretation at one point in time. A learned model can discover the underlying geometry from examples without being locked to specific threshold values, and may generalize better across market regimes.

Why include XGBoost as a classical ML baseline? On this dataset, effective sample size is ~86 independent windows (due to 60-bar stride overlap). In small-data regimes, gradient boosting with hand-crafted features often outperforms DL. XGBoost is here as an honest control, not a fallback.

---

## 3. Data: building a labeled dataset from scratch

No public labeled FVG dataset exists, so everything starts from raw price data.

### Source: Alpaca Markets free tier

The project pulls SPY 1-minute bars from Alpaca's API, covering 2016 to 2025. Those 1-minute bars are resampled to hourly, anchored at 09:30 ET, with Regular Trading Hours filtering (09:30-15:59 ET only).

Two sources were evaluated:

| Source | Verdict | Reason |
|--------|---------|--------|
| Alpaca free tier | Used | Full history to 2016, `adjustment="raw"` available |
| yfinance | Rejected | Hard-capped at 730 days for H1 history (confirmed empirically, commit `8f02b3d`) |

The `adjustment="raw"` setting is deliberate. SMC analysis depends on actual printed prices, dividend adjustments distort the geometric relationships that define FVG zones.

### Two labellers: raw vs valid

Two separate labellers were built and kept in the codebase:

`FVGLabeller` (raw) applies pure 3-candle geometry with the label placed at N+1 (the first bar where the pattern is fully confirmed). It fires on ~25% of bars.

`ValidFVGLabeller` (canonical) adds the 6 SMC criteria on top. It fires on ~3% of bars, and is the actual training target.

The label index is not a minor detail. An FVG is a 3-candle pattern spanning bars N-1, N, and N+1. The pattern is only knowable after N+1 closes, so the label must sit at N+1, not N. Any label placed at N uses future data.

A popular Python library (`smartmoneyconcepts` v0.0.27) was audited and rejected: source code inspection confirmed it places FVG labels at bar N using `shift(-1)`, which is exactly one bar of lookahead. GitHub PR #95 to fix this was open and unmerged at the time of review (research artifact: `.nb-suite/research/8-May-26/smc-library-validation.md`). Custom implementation used instead.

### Gold annotation: validating the labeller

200 bars were annotated by hand, stratified across FVG-rich, low-volatility, and random windows. The `ValidFVGLabeller` achieved Cohen's κ = 1.0 against the human annotations. The raw `FVGLabeller` got κ = 0.28, confirming that pure geometry is a poor proxy for the FVG concept as traders actually apply it.

### Dataset splits (temporal, no shuffle)

| Split | Dates | Bars | None | Bull FVG | Bear FVG |
|-------|-------|------|------|----------|----------|
| Train | 2016-01-04 to 2021-12-31 | 10,577 | 96.8% | 1.9% | 1.3% |
| Val   | 2022-01-03 to 2022-12-30 | 1,757  | 96.3% | 1.5% | 2.2% |
| Test  | 2023-01-03 to 2025-12-30 | 5,257  | 96.9% | 1.8% | 1.3% |

All splits are strictly chronological. The test set was not accessed during hyperparameter search or model selection.

### Pipeline: from raw ticks to model-ready windows

The data journey from Alpaca API to a training-ready window tensor has five deterministic steps. Every transformation preserves temporal order and nothing leaks from future bars into past labels. Figure 6 walks through each stage with bar counts.


In [3]:
"""
Figure 6 — Data pipeline: Alpaca 1-min → H1 RTH → labels → split → windows.
Static Plotly figure — no live data required.
"""

import plotly.graph_objects as go

stages = [
    ("Alpaca pull<br>1-min bars<br>2016-2025", "~2.3 M bars<br>raw ticks"),
    ("RTH filter<br>+ resample H1<br>(09:30-15:59 ET)", "~17,000 bars<br>hourly"),
    ("ValidFVG<br>labeller<br>(6 criteria)", "17,591 bars<br>~3% positive"),
    ("Temporal split<br>Train/Val/Test<br>(no shuffle)", "10 577 / 1 757 / 5 257"),
    ("Sliding window<br>W=60, stride=1<br>+ normalise", "~17,500 windows<br>(train+val+test)"),
    ("Model input<br>(60, 5) tensor<br>or 35 features", "Ready for training"),
]

colors = ["#60a5fa", "#34d399", "#34d399", "#f59e0b", "#a78bfa", "#e2e8f0"]
x = list(range(len(stages)))

fig = go.Figure()

for i, (label, count) in enumerate(stages):
    fig.add_shape(
        type="rect",
        x0=i - 0.38, x1=i + 0.38,
        y0=0.3, y1=0.7,
        fillcolor=colors[i],
        line_color="white", line_width=2,
        layer="below",
    )
    fig.add_annotation(
        x=i, y=0.5,
        text=f"<b>{label}</b>",
        font=dict(size=11, color="white"),
        showarrow=False,
        align="center",
    )
    fig.add_annotation(
        x=i, y=0.22,
        text=f"<i>{count}</i>",
        font=dict(size=10, color="#374151"),
        showarrow=False,
        align="center",
    )
    if i < len(stages) - 1:
        fig.add_annotation(
            x=i + 0.5, y=0.5,
            text="→",
            font=dict(size=18, color="#94a3b8"),
            showarrow=False,
        )

fig.update_layout(
    title={
        "text": ("Figure 6. Data pipeline from Alpaca raw feed to model-ready tensors.<br>"
                 "<sup>All transforms deterministic and temporal-order-preserving | Zero lookahead</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    xaxis=dict(visible=False, range=[-0.5, len(stages) - 0.5]),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=280,
    margin=dict(t=90, b=50, l=10, r=10),
    font=dict(size=12),
)

# fig.write_html("../reports/figure06_pipeline.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure06_pipeline.html")


*Figure 6. Each stage reduces or reshapes the data without touching its temporal structure. The key design constraint is that the label at any bar N+1 is computed only from bars up to and including N+1 -- nothing beyond. The 1-minute pull is large but ephemeral; downstream artifacts are the 17,591-bar H1 parquet and the label column, both versioned in the repo.*


In [4]:
"""
Figure 7 — Temporal split timeline (2016-2025).
"""

import plotly.graph_objects as go

split_data = [
    ("Train", 2016.0, 2022.0, "#3b82f6", "10,577 bars"),
    ("Val",   2022.0, 2023.0, "#f59e0b", "1,757 bars"),
    ("Test",  2023.0, 2026.0, "#10b981", "5,257 bars"),
]

fig = go.Figure()

for label, start, end, color, count in split_data:
    fig.add_shape(
        type="rect",
        x0=start, x1=end, y0=0.2, y1=0.8,
        fillcolor=color,
        opacity=0.85,
        line_color="white", line_width=2,
    )
    mid = (start + end) / 2
    fig.add_annotation(x=mid, y=0.5, text=f"<b>{label}</b><br>{count}",
                       font=dict(size=13, color="white"), showarrow=False, align="center")

# Boundary annotations
for boundary, label in [(2022.0, "Jan 2022"), (2023.0, "Jan 2023")]:
    fig.add_shape(type="line", x0=boundary, x1=boundary, y0=0.15, y1=0.85,
                  line=dict(color="white", width=2, dash="dot"))
    fig.add_annotation(x=boundary, y=0.1, text=label,
                       font=dict(size=11, color="#374151"), showarrow=False)

fig.update_layout(
    title={
        "text": ("Figure 7. Temporal train/val/test split — strictly chronological, no shuffle.<br>"
                 "<sup>Test set never accessed during HP search or model selection</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    xaxis=dict(range=[2015.5, 2026.2], tickvals=list(range(2016, 2027)),
               ticktext=[str(y) for y in range(2016, 2027)],
               showgrid=False, zeroline=False),
    yaxis=dict(visible=False, range=[0, 1]),
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=230,
    margin=dict(t=90, b=60, l=20, r=20),
    font=dict(size=12),
)

# fig.write_html("../reports/figure07_temporal_split.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure07_temporal_split.html")


*Figure 7. The train set covers 2016-2021 (6 years of mixed regimes including COVID volatility). Validation is a single year (2022) used only for hyperparameter selection. Test is 2023-2025, held out completely until final evaluation. There is no shuffling at any stage -- a model that can peek at future bars is not a real model.*


### Sliding window: how each bar becomes a training sample

Every model receives a **60-bar window** of OHLCV data ending at the bar to be labeled. With stride=1, adjacent windows overlap by 59 bars -- this is why effective independent sample count (~86) is so much smaller than the raw window count (~5,200 in test). Figure 8 illustrates the sliding window on a real SPY price segment.


In [5]:
"""
Figure 8 — Sliding 60-bar window concept on synthetic SPY-like prices.
Shows 3 overlapping windows, stride=1.
"""

import plotly.graph_objects as go
import numpy as np

np.random.seed(42)
n_bars = 75
prices = 440 + np.cumsum(np.random.randn(n_bars) * 0.4)
x = list(range(n_bars))

fig = go.Figure()

# Full price line
fig.add_trace(go.Scatter(
    x=x, y=prices,
    mode="lines",
    line=dict(color="#94a3b8", width=1.5),
    name="SPY H1 price",
    showlegend=True,
))

# Three overlapping windows
windows = [
    (0,  60, "#3b82f6", "Window i (bars 0-59, label at bar 59)"),
    (5,  65, "#f59e0b", "Window i+5 (bars 5-64, label at bar 64)"),
    (10, 70, "#10b981", "Window i+10 (bars 10-69, label at bar 69)"),
]

for start, end, color, name in windows:
    fig.add_vrect(
        x0=start, x1=end,
        fillcolor=color, opacity=0.10,
        line_color=color, line_width=1.5,
        annotation_text="",
    )
    # Label position marker
    label_bar = end - 1
    fig.add_trace(go.Scatter(
        x=[label_bar], y=[prices[label_bar]],
        mode="markers",
        marker=dict(size=10, color=color, symbol="diamond"),
        name=name,
        showlegend=True,
    ))

fig.update_layout(
    title={
        "text": ("Figure 8. Sliding 60-bar window with stride=1 on SPY H1.<br>"
                 "<sup>Diamond = label bar (N+1) | Adjacent windows overlap by 59 bars | "
                 "Effective independent samples ≈ window_size = 60</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    xaxis_title="Bar index",
    yaxis_title="Price (USD)",
    plot_bgcolor="white",
    height=400,
    margin=dict(t=110, b=60),
    font=dict(size=12),
    legend=dict(orientation="h", y=-0.2),
)

# fig.write_html("../reports/figure08_window_concept.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure08_window_concept.html")


*Figure 8. Three overlapping windows (stride=1) on the same price segment. Diamond markers show the label position at bar 60 (the last bar of each window). Because consecutive windows share 59 of 60 bars, their predictions are correlated -- the effective independent sample count is roughly n_bars / window_size, not n_bars. This is why the G10 bootstrap uses block-size=60 rather than naive i.i.d. resampling.*


---

## 4. Models: what was tried and why

| Model | Why included | Mean Macro F1 (5-seed) | Std | 95% CI |
|-------|-------------|----------------------|-----|--------|
| Naive majority | "Always predict None" sanity check | 0.328 |, |, |
| XGBoost | Classical baseline; hand-crafted features strong in small-data regime | **0.721** | 0.001 | [0.671, 0.761] |
| CNN-LSTM | Conv kernel=3/5 encodes 3-candle FVG locality directly | 0.614 | 0.021 | [0.576, 0.649] |
| LSTM | Standard sequence model for OHLCV time series | 0.599 | 0.025 | [0.563, 0.635] |

All models were trained on the same splits. Primary metric is macro F1 on the test set, averaged across 5 random seeds. The naive majority model matters here: on a 97/2/1 class split, accuracy is meaningless. A model that always predicts "none" gets 97% accuracy but only F1 = 0.328.

XGBoost won by a large margin. This is the central finding so far. Classical ML with 35 hand-crafted features outperforms both DL models by 0.107-0.122 F1 points. This isn't a failure of DL, it's expected on small structured data. Effective sample size is ~86 independent windows, and gradient boosting handles that regime well. DL typically needs more samples to express its inductive biases.

The honest read: for this label, this dataset size, and this feature representation, XGBoost is currently the best model. The DL ladder continues, xLSTM and Transformer are untested, but the bar is clearly set.

### Input representation

Each model receives a 60-bar sliding window of OHLCV data (open, high, low, close, volume), representing roughly 10 trading days.

LSTM and CNN-LSTM take the raw window as a `(60, 5)` tensor, normalised per-window with min-max scaling. XGBoost receives 35 hand-crafted features extracted from the same window, momentum, gap geometry, volatility, session position.

### Architecture notes

CNN-LSTM uses two Conv1d layers (16 filters, kernel=5) feeding into a 2-layer LSTM (hidden=32). The conv layers act as local pattern detectors before the recurrent layers process the sequence. Kernel size 5 was chosen because the FVG pattern spans exactly 3 candles, a kernel of 5 captures that neighborhood with one position of margin on each side.

The standard LSTM (single unidirectional layer, hidden=128) is weaker here partly because raw OHLCV without a convolutional front-end must learn local patterns purely from recurrent state, which is less sample-efficient than explicit local filtering.

In [6]:
"""
Figure 2 — Model F1 comparison with error bars.
Reads from JSON artifacts — no model re-training.
"""
import json, os
import plotly.graph_objects as go

BASE = "../reports/rigor/2026-05-13"

# Load multiseed summaries
with open(f"{BASE}/G2/multiseed_summary.json") as f:
    g2 = json.load(f)
with open(f"{BASE}/cnn_lstm_G2/multiseed_summary.json") as f:
    cnn_g2 = json.load(f)

models = ["Naive Majority", "LSTM", "CNN-LSTM", "XGBoost"]
means  = [0.328,             g2["lstm"]["mean_macro_f1"], cnn_g2["mean_test_macro_f1"], g2["xgb"]["mean_macro_f1"]]
errors = [0,                 g2["lstm"]["std_macro_f1"],  cnn_g2["std_test_macro_f1"],  g2["xgb"]["std_macro_f1"]]
colors = ["#9ca3af", "#60a5fa", "#34d399", "#f59e0b"]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=models, y=means,
    error_y=dict(type="data", array=errors, visible=True),
    marker_color=colors,
    text=[f"{m:.3f}" for m in means],
    textposition="outside",
))
fig.update_layout(
    title={
        "text": (
            "Figure 2. Test macro F1 by model (5-seed mean ± std).<br>"
            "<sup>ValidFVG labels | SPY H1 2016-2025 | Test: 2023-2025 | Temporal split</sup>"
        ),
        "x": 0.5, "xanchor": "center"
    },
    yaxis=dict(title="Macro F1", range=[0, 0.85]),
    xaxis_title="Model",
    plot_bgcolor="white",
    height=420,
    margin=dict(t=100, b=60),
    font=dict(size=13),
)
fig.add_hline(y=0.328, line_dash="dot", line_color="#9ca3af", annotation_text="Naive baseline", annotation_position="right")

os.makedirs("../reports", exist_ok=True)
# fig.write_html("../reports/figure02_model_f1_comparison.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure02_model_f1_comparison.html")

*Figure 2. XGBoost (0.721 ± 0.001) leads by a wide margin over CNN-LSTM (0.614 ± 0.021) and LSTM (0.599 ± 0.025). Error bars show standard deviation across 5 seeds. XGBoost's near-zero std reflects bit-exact reproducibility across seeds, the model is fully deterministic at this dataset size. LSTM has the highest variance, driven by sensitivity to weight initialization on small effective sample counts.*

---

## 5. Hyperparameter search and ablations

Each model went through a 10-gap rigor process before its results were accepted. The gaps cover the most common sources of inflated reported numbers in ML work.

### G1, Hyperparameter tuning (Optuna)

Optuna ran automated HP search with a 60-minute wall-clock budget per model.

LSTM best HP (38 completed + 38 pruned trials, best val F1 = 0.637, trial 42):

| Param | Value |
|-------|-------|
| hidden_size | 128 |
| num_layers | 1 |
| dropout | 0.318 |
| head_dropout | 0.526 |
| lr | 5.30e-4 |
| batch_size | 16 |

XGBoost best HP (50 trials, best val F1 = 0.691, trial 31):

| Param | Value |
|-------|-------|
| n_estimators | 513 |
| max_depth | 4 |
| learning_rate | 0.1311 |
| subsample | 0.826 |

CNN-LSTM best HP (6 trials, 60 min cap, best val F1 = 0.644, trial 0):

| Param | Value |
|-------|-------|
| n_conv_layers | 2 |
| conv_filters | 16 |
| kernel_size | 5 |
| lstm_hidden | 32 |
| lstm_layers | 2 |
| dropout | 0.2217 |

### Key ablation findings (G3-G9)

| Gap | What was tested | Finding |
|-----|----------------|---------|
| G3 | Focal loss γ ∈ {1,2,3} vs WeightedCE | Focal uniformly worse: γ=1 gives -0.064, γ=3 gives -0.109 |
| G4 | Decision threshold tuning via PR-curve | Negligible or harmful: LSTM -0.006, XGB -0.043 |
| G5 | SHAP feature importance (XGB) | `ret_60` (0.838) and `gap_norm_bull` (0.814) dominate; model learns from the right signals |
| G6 | Bull vs bear prediction asymmetry | +0.035 bull>bear gap, not systematic across seeds |
| G7 | Window size sweep W ∈ {30,45,60,90,120} | W=60 balances F1 (0.617) and sample coverage (eff_n=86); W=90 marginally better F1 but only 57 independent samples |
| G8 | Data scaling: 2018-2024 vs 2016-2025 | Extending data gave negligible change; max delta = -0.013 |
| G9 | Regularisation ablation (dropout + L2) | Head dropout is primary regulariser; removing both drops LSTM by -0.020 |

One finding worth flagging explicitly: focal loss is a standard recommendation for class-imbalanced problems, so it was predicted to help. It didn't, at ~3% positive rate, the loss is already dominated by hard minority examples, and adding focal weighting pushes gradient noise into instability. WeightedCE handles this label distribution better.

In [7]:
"""
Figure 9 — Per-class F1: None / Bull FVG / Bear FVG x XGB / LSTM / CNN-LSTM.
"""

import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G10/bootstrap_ci_xgb.json") as f:
    xgb = json.load(f)

with open("../reports/rigor/2026-05-13/G10/bootstrap_ci_lstm.json") as f:
    lstm = json.load(f)

with open("../reports/rigor/2026-05-13/cnn_lstm_G10/bootstrap_ci_cnn_lstm.json") as f:
    cnn = json.load(f)

# XGB + CNN: flat schema. LSTM: nested schema
xgb_macro = xgb["macro_f1_mean"]; xgb_bull = xgb["bull_f1_mean"]; xgb_bear = xgb["bear_f1_mean"]
xgb_none = round(xgb_macro * 3 - xgb_bull - xgb_bear, 3)

lstm_macro = lstm["macro_f1"]["point"]; lstm_bull = lstm["bull_f1"]["point"]; lstm_bear = lstm["bear_f1"]["point"]
lstm_none = round(lstm_macro * 3 - lstm_bull - lstm_bear, 3)

cnn_macro = cnn["macro_f1_mean"]; cnn_bull = cnn["bull_f1_mean"]; cnn_bear = cnn["bear_f1_mean"]
cnn_none = round(cnn_macro * 3 - cnn_bull - cnn_bear, 3)

classes = ["None (majority)", "Bull FVG", "Bear FVG"]
data = {
    "XGBoost":  ([xgb_none,  round(xgb_bull,3),  round(xgb_bear,3)],  "#3b82f6"),
    "LSTM":     ([lstm_none, round(lstm_bull,3), round(lstm_bear,3)], "#f97316"),
    "CNN-LSTM": ([cnn_none,  round(cnn_bull,3),  round(cnn_bear,3)],  "#22c55e"),
}

fig = go.Figure()
for name, (vals, color) in data.items():
    fig.add_trace(go.Bar(
        name=name, x=classes, y=vals,
        marker_color=color,
        text=[f"{v:.3f}" for v in vals],
        textposition="outside",
    ))

fig.update_layout(
    barmode="group",
    title={
        "text": ("Figure 9. Per-class F1 — all three models struggle on minority classes.<br>"
                 "<sup>None class drives macro F1 | XGBoost leads on Bull and Bear FVG | Test set 2023-2025</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    yaxis=dict(title="F1 Score", range=[0, 1.1]),
    plot_bgcolor="white",
    height=420,
    margin=dict(t=110, b=60),
    font=dict(size=12),
    legend=dict(orientation="h", y=-0.18),
)

# fig.write_html("../reports/figure09_perclass_f1.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure09_perclass_f1.html")


*Figure 9. All three models achieve F1 > 0.96 on the None class -- not useful, since that class represents 97% of bars. The real signal is in Bull and Bear FVG. XGBoost leads on both (0.601 bull, 0.580 bear). CNN-LSTM and LSTM are competitive with each other but trail XGBoost by roughly 0.15 F1 points per minority class. The bear FVG gap between DL and XGBoost (0.42 vs 0.58) is larger than the bull gap, suggesting the 3-candle convolutional approach captures bullish momentum more readily than bearish reversal.*


In [8]:
"""
Figure 10 — Bootstrap 95% CI forest plot for macro F1 (all 3 models).
Block-bootstrap CI: block_size=60, n_bootstrap=1000.
"""

import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G10/bootstrap_ci_xgb.json") as f:
    xgb = json.load(f)

with open("../reports/rigor/2026-05-13/G10/bootstrap_ci_lstm.json") as f:
    lstm = json.load(f)

with open("../reports/rigor/2026-05-13/cnn_lstm_G10/bootstrap_ci_cnn_lstm.json") as f:
    cnn = json.load(f)

# Unified access — handle both flat (xgb, cnn) and nested (lstm) schemas
def get_macro(data):
    if "macro_f1_mean" in data:
        return data["macro_f1_mean"], data["ci_95_lower"], data["ci_95_upper"]
    return data["macro_f1"]["point"], data["macro_f1"]["ci_lower"], data["macro_f1"]["ci_upper"]

xgb_pt,  xgb_lo,  xgb_hi  = get_macro(xgb)
lstm_pt, lstm_lo, lstm_hi  = get_macro(lstm)
cnn_pt,  cnn_lo,  cnn_hi   = get_macro(cnn)

models = ["XGBoost", "CNN-LSTM", "LSTM"]
points = [xgb_pt, cnn_pt, lstm_pt]
lowers = [xgb_lo, cnn_lo, lstm_lo]
uppers = [xgb_hi, cnn_hi, lstm_hi]
colors = ["#3b82f6", "#22c55e", "#f97316"]

fig = go.Figure()

for i, (model, pt, lo, hi, color) in enumerate(zip(models, points, lowers, uppers, colors)):
    fig.add_trace(go.Scatter(
        x=[lo, hi], y=[model, model],
        mode="lines",
        line=dict(color=color, width=3),
        showlegend=False,
        hoverinfo="skip",
    ))
    for cap_x in [lo, hi]:
        fig.add_shape(type="line", x0=cap_x, x1=cap_x,
                      y0=i - 0.15, y1=i + 0.15,
                      line=dict(color=color, width=2),
        )
    fig.add_trace(go.Scatter(
        x=[pt], y=[model],
        mode="markers+text",
        marker=dict(size=14, color=color, symbol="circle"),
        text=[f"  {pt:.3f}"],
        textposition="middle right",
        textfont=dict(size=12),
        name=model,
    ))

fig.add_vline(x=0.328, line_dash="dot", line_color="#9ca3af",
              annotation_text="Naive (0.328)", annotation_position="top right")

fig.update_layout(
    title={
        "text": ("Figure 10. Bootstrap 95% CI for macro F1 — no CI overlap between XGBoost and DL.<br>"
                 "<sup>Block-bootstrap, block_size=60, n=1000 | Effective independent test windows = 86</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    xaxis=dict(title="Macro F1 (higher = better)", range=[0.45, 0.85]),
    yaxis=dict(title=""),
    plot_bgcolor="white",
    height=320,
    margin=dict(t=110, b=60, l=100, r=120),
    font=dict(size=12),
    showlegend=False,
)

# fig.write_html("../reports/figure10_bootstrap_ci.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure10_bootstrap_ci.html")


*Figure 10. Block-bootstrap 95% confidence intervals on the test set. The CIs for XGBoost [0.671, 0.761] and CNN-LSTM [0.576, 0.649] do not overlap, which means the 0.107 F1 gap between them is statistically distinguishable given the 86 effective independent samples available. The wide intervals reflect real uncertainty -- with only ~86 independent test windows, all point estimates carry substantial sampling noise. This is why bootstrap CI is reported alongside seed variance.*


In [9]:
"""
Figure 11 -- Focal loss ablation: gamma in {1,2,3} vs WeightedCE baseline (LSTM).
"""

import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G3/focal_ablation.json") as f:
    focal = json.load(f)

labels  = ["WeightedCE (baseline)", "Focal g=1", "Focal g=2", "Focal g=3"]
means   = [
    focal["weighted_ce_baseline"]["mean_macro_f1"],
    focal["focal_gamma_1"]["mean_macro_f1"],
    focal["focal_gamma_2"]["mean_macro_f1"],
    focal["focal_gamma_3"]["mean_macro_f1"],
]
stds    = [
    focal["weighted_ce_baseline"]["std_macro_f1"],
    focal["focal_gamma_1"]["std_macro_f1"],
    focal["focal_gamma_2"]["std_macro_f1"],
    focal["focal_gamma_3"]["std_macro_f1"],
]
deltas  = [0] + [focal[f"focal_gamma_{g}"]["delta_vs_baseline"] for g in [1,2,3]]
colors  = ["#3b82f6"] + ["#ef4444"] * 3

fig = go.Figure()
fig.add_trace(go.Bar(
    x=labels, y=means,
    error_y=dict(type="data", array=stds, visible=True),
    marker_color=colors,
    text=[f"{m:.3f} ({d:+.3f})" for m, d in zip(means, deltas)],
    textposition="outside",
))
fig.add_hline(y=means[0], line_dash="dot", line_color="#3b82f6",
              annotation_text="WeightedCE baseline", annotation_position="right")

fig.update_layout(
    title={
        "text": (
            "Figure 11. Focal loss makes things worse -- WeightedCE wins at every gamma (LSTM, 5-seed mean).<br>"
            "<sup>All gamma tested with same G1 HP | delta = delta vs WeightedCE baseline</sup>"
        ),
        "x": 0.5, "xanchor": "center",
    },
    yaxis=dict(title="Macro F1 (higher = better)", range=[0.40, 0.68]),
    plot_bgcolor="white",
    height=420,
    margin=dict(t=110, b=60, r=140),
    font=dict(size=12),
    showlegend=False,
)

# fig.write_html("../reports/figure11_focal_ablation.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure11_focal_ablation.html")


*Figure 11. The common recommendation for class-imbalanced problems is focal loss (Lin et al., 2017), which up-weights hard misclassified examples. Here it hurts at every gamma value: gamma=1 costs 0.064 F1, gamma=3 costs 0.109. The reason is the label distribution: at ~3% positive rate, WeightedCE already concentrates gradient on hard minority examples via explicit class weights. Adding focal modulation on top amplifies noise from the hardest minority examples, destabilising training.*


In [10]:
"""
Figure 12 — Window size sweep: W=30,45,60,90,120 vs val + test F1 (LSTM).
Data: reports/rigor/2026-05-13/G7/window_sweep_results.json
"""

import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G7/window_sweep_results.json") as f:
    sweep = json.load(f)

ws   = [r["window_size"] for r in sweep]
val  = [r["val_macro_f1"] for r in sweep]
test = [r["test_macro_f1"] for r in sweep]
eff  = [r["effective_n"] for r in sweep]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=ws, y=val,
    mode="lines+markers",
    name="Val F1",
    line=dict(color="#f97316", width=2),
    marker=dict(size=8),
    hovertemplate="W=%{x}, val F1=%{y:.3f}<extra></extra>",
))
fig.add_trace(go.Scatter(
    x=ws, y=test,
    mode="lines+markers",
    name="Test F1",
    line=dict(color="#3b82f6", width=2, dash="dash"),
    marker=dict(size=8),
    hovertemplate="W=%{x}, test F1=%{y:.3f}<extra></extra>",
))

# Annotate effective N
for w, e in zip(ws, eff):
    fig.add_annotation(x=w, y=min(val[ws.index(w)], test[ws.index(w)]) - 0.018,
                       text=f"eff_n={e}", font=dict(size=10, color="#6b7280"), showarrow=False)

# Highlight W=60
fig.add_vline(x=60, line_dash="dot", line_color="#22c55e",
              annotation_text="W=60 (chosen)", annotation_position="top right")

fig.update_layout(
    title={
        "text": ("Figure 12. Window size vs F1 — W=60 balances performance and independent sample count.<br>"
                 "<sup>LSTM, seed42 | eff_n = effective independent test windows (approx n_bars / W)</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    xaxis=dict(title="Window size (bars)", tickvals=ws),
    yaxis=dict(title="Macro F1 (higher = better)", range=[0.48, 0.70]),
    plot_bgcolor="white",
    height=420,
    margin=dict(t=110, b=70, r=100),
    font=dict(size=12),
    legend=dict(orientation="h", y=-0.22),
)

# fig.write_html("../reports/figure12_window_sweep.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure12_window_sweep.html")


*Figure 12. Val F1 improves steadily from W=30 (0.557) to W=120 (0.656), but test F1 peaks at W=60 (0.617) and drops at W=120 (0.579). W=90 has a slightly higher test F1 (0.633) but only 57 effective independent test windows -- a 34% reduction vs W=60's 86 windows. W=60 was selected as the best trade-off between signal richness and sample independence. The val-test divergence at W=120 is a mild overfitting signal: the model starts memorising validation-specific temporal patterns.*


In [11]:
"""
Figure 13 -- Regularisation ablation: control / no-dropout / no-L2 / both-off.
LSTM from G9. CNN-LSTM from cnn_lstm_G9 md files.
"""

import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G9/reg_ablation_summary.json") as f:
    lstm_reg = json.load(f)

# CNN-LSTM G9: no combined JSON; values from md summaries
cnn_control  = 0.6141   # G2 mean (full reg)
cnn_no_drop  = 0.6227   # cnn_lstm_G9/no_dropout md
cnn_no_l2    = 0.5803   # cnn_lstm_G9/no_l2 md

configs = ["Control (full reg)", "No dropout", "No L2", "Both off"]
lstm_vals = [
    lstm_reg["configs"]["control"]["mean_macro_f1"],
    lstm_reg["configs"]["no_dropout"]["mean_macro_f1"],
    lstm_reg["configs"]["no_l2"]["mean_macro_f1"],
    lstm_reg["configs"]["both_off"]["mean_macro_f1"],
]
lstm_delta = [0] + [
    lstm_reg["configs"]["no_dropout"]["delta_vs_control"],
    lstm_reg["configs"]["no_l2"]["delta_vs_control"],
    lstm_reg["configs"]["both_off"]["delta_vs_control"],
]
cnn_vals  = [cnn_control, cnn_no_drop, cnn_no_l2, None]

fig = go.Figure()

fig.add_trace(go.Bar(
    name="LSTM",
    x=configs, y=lstm_vals,
    marker_color="#f97316",
    text=[f"{v:.3f} ({d:+.3f})" for v, d in zip(lstm_vals, lstm_delta)],
    textposition="outside",
))

cnn_display = [v if v is not None else 0 for v in cnn_vals]
cnn_text    = [f"{v:.3f}" if v is not None else "n/a" for v in cnn_vals]
fig.add_trace(go.Bar(
    name="CNN-LSTM",
    x=configs, y=cnn_display,
    marker_color="#22c55e",
    text=cnn_text,
    textposition="outside",
))

fig.update_layout(
    barmode="group",
    title={
        "text": (
            "Figure 13. Removing dropout hurts more than removing L2 -- dropout is the primary regulariser.<br>"
            "<sup>LSTM (5-seed) + CNN-LSTM (3 seeds, both-off not run) | G9 ablation</sup>"
        ),
        "x": 0.5, "xanchor": "center",
    },
    yaxis=dict(title="Macro F1 (higher = better)", range=[0.52, 0.68]),
    plot_bgcolor="white",
    height=430,
    margin=dict(t=110, b=60),
    font=dict(size=12),
    legend=dict(orientation="h", y=-0.18),
)

# fig.write_html("../reports/figure13_reg_ablation.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure13_reg_ablation.html")


*Figure 13. For LSTM, removing dropout drops F1 by 0.018 (control: 0.601, no-dropout: 0.583). Removing L2 alone is milder (-0.008). Removing both simultaneously costs 0.020, similar to dropout alone -- this suggests head dropout is the dominant regulariser and L2 provides marginal benefit beyond it. For CNN-LSTM, the no-dropout condition slightly improves performance (0.622 vs 0.614 control), possibly because CNN-LSTM's convolutional layers already act as implicit smoothers, making explicit dropout partially redundant. The no-L2 condition hurts more for CNN-LSTM (-0.034 vs control) than for LSTM (-0.008).*


In [12]:
"""
Figure 14 -- CNN-LSTM Optuna trial scatter: val F1 vs trial number, coloured by lstm_hidden.
"""

import optuna
import plotly.graph_objects as go

optuna.logging.set_verbosity(optuna.logging.WARNING)

storage = "sqlite:///../checkpoints/cnn_lstm/optuna_2026-05-12.db"
study = optuna.load_study(study_name="cnn_lstm_fvg", storage=storage)

trials = [(t.number, t.value, t.params) for t in study.trials if t.state.name == "COMPLETE" and t.value]
trial_ns     = [t[0] for t in trials]
trial_f1s    = [t[1] for t in trials]
trial_hidden = [t[2].get("lstm_hidden", 32) for t in trials]
trial_kernel = [t[2].get("kernel_size", 5) for t in trials]
trial_text   = [
    f"Trial {t[0]}<br>val F1={t[1]:.4f}<br>lstm_hidden={t[2].get('lstm_hidden')}<br>kernel={t[2].get('kernel_size')}"
    for t in trials
]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=trial_ns, y=trial_f1s,
    mode="markers+text",
    marker=dict(
        size=14,
        color=trial_hidden,
        colorscale="Blues",
        colorbar=dict(title="lstm_hidden"),
        showscale=True,
        symbol=["circle" if k == 5 else "square" for k in trial_kernel],
    ),
    text=[f"T{n}" for n in trial_ns],
    textposition="top center",
    hovertext=trial_text,
    hoverinfo="text",
    showlegend=False,
))

best_n = trial_ns[trial_f1s.index(max(trial_f1s))]
best_f1 = max(trial_f1s)
fig.add_annotation(x=best_n, y=best_f1 + 0.004,
                   text=f"Best: T{best_n} val F1={best_f1:.3f}",
                   font=dict(size=11, color="#1e40af"), showarrow=True,
                   arrowhead=2, arrowcolor="#1e40af")

fig.update_layout(
    title={
        "text": (
            "Figure 14. CNN-LSTM Optuna trials -- best HP found early (Trial 0, 60-min budget).<br>"
            "<sup>Circle = kernel_size=5 | Square = other kernel | Colour = lstm_hidden size | 6 complete trials</sup>"
        ),
        "x": 0.5, "xanchor": "center",
    },
    xaxis=dict(title="Trial number", dtick=1),
    yaxis=dict(title="Val Macro F1 (higher = better)", range=[0.47, 0.67]),
    plot_bgcolor="white",
    height=420,
    margin=dict(t=110, b=60, r=120),
    font=dict(size=12),
)

# fig.write_html("../reports/figure14_optuna_cnn_lstm.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure14_optuna_cnn_lstm.html")


*Figure 14. The 60-minute Optuna budget produced only 6 complete CNN-LSTM trials -- compared to 38 complete + 38 pruned for LSTM. Trial 0 happened to be the best configuration (val F1 = 0.644): 2 conv layers, 16 filters, kernel=5, 2-layer LSTM with hidden=32. Subsequent trials with more conv layers, more filters, or larger LSTM hidden sizes (trials 1-2 with hidden=32/128) scored lower. The small trial count means the HP search is far from exhaustive, and better configurations likely exist. This is a direct consequence of the MPS training ban adding ~2x CPU overhead per trial.*


### Why three different architectures?

Each model receives the same underlying data but processes it in a fundamentally different way. Figure 15 illustrates the input transformation for each approach.


In [13]:
"""
Figure 15 — Model input transformation cartoon: XGB / LSTM / CNN-LSTM.
Static Plotly figure — no model loading required.
"""

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["XGBoost", "LSTM", "CNN-LSTM"],
    horizontal_spacing=0.08,
)

def add_box(fig, row, col, x, y, w, h, text, color, fontsize=11):
    fig.add_shape(type="rect", x0=x, x1=x+w, y0=y, y1=y+h,
                  fillcolor=color, line_color="white", line_width=1.5,
                  row=row, col=col)
    fig.add_annotation(x=x+w/2, y=y+h/2, text=text,
                       font=dict(size=fontsize, color="white"),
                       showarrow=False, align="center",
                       row=row, col=col)

def add_arrow(fig, row, col, x, y1, y2):
    fig.add_annotation(x=x, y=(y1+y2)/2, text="↓",
                       font=dict(size=18, color="#94a3b8"),
                       showarrow=False, row=row, col=col)

# ---- XGBoost ----
add_box(fig, 1, 1, 0.1, 0.75, 0.8, 0.15, "60-bar window<br>(60×5 OHLCV)", "#60a5fa")
add_arrow(fig, 1, 1, 0.5, 0.75, 0.6)
add_box(fig, 1, 1, 0.1, 0.45, 0.8, 0.15, "Feature extractor<br>35 hand-crafted signals", "#3b82f6")
add_arrow(fig, 1, 1, 0.5, 0.45, 0.3)
add_box(fig, 1, 1, 0.1, 0.15, 0.8, 0.15, "Gradient boosted trees<br>→ class probabilities", "#1d4ed8")

# ---- LSTM ----
add_box(fig, 1, 2, 0.1, 0.75, 0.8, 0.15, "60-bar window<br>(60×5 normalised)", "#f97316")
add_arrow(fig, 1, 2, 0.5, 0.75, 0.6)
add_box(fig, 1, 2, 0.1, 0.45, 0.8, 0.15, "2-layer LSTM<br>hidden=128, unidirectional", "#ea580c")
add_arrow(fig, 1, 2, 0.5, 0.45, 0.3)
add_box(fig, 1, 2, 0.1, 0.15, 0.8, 0.15, "Linear head<br>→ class probabilities", "#c2410c")

# ---- CNN-LSTM ----
add_box(fig, 1, 3, 0.1, 0.75, 0.8, 0.12, "60-bar window<br>(60×5 normalised)", "#22c55e")
add_arrow(fig, 1, 3, 0.5, 0.75, 0.62)
add_box(fig, 1, 3, 0.1, 0.50, 0.8, 0.12, "2×Conv1d<br>filters=16, kernel=5", "#16a34a")
add_arrow(fig, 1, 3, 0.5, 0.50, 0.37)
add_box(fig, 1, 3, 0.1, 0.25, 0.8, 0.12, "2-layer LSTM<br>hidden=32", "#15803d")
add_arrow(fig, 1, 3, 0.5, 0.25, 0.12)
add_box(fig, 1, 3, 0.1, 0.00, 0.8, 0.12, "Linear head<br>→ class probabilities", "#166534")

for col in [1, 2, 3]:
    fig.update_xaxes(visible=False, range=[0, 1], row=1, col=col)
    fig.update_yaxes(visible=False, range=[-0.1, 1.0], row=1, col=col)

fig.update_layout(
    title={
        "text": ("Figure 15. Input transformation per model: hand-crafted features vs raw sequence vs conv-then-recurrent.<br>"
                 "<sup>XGBoost encodes domain knowledge | LSTM learns patterns from raw OHLCV | "
                 "CNN-LSTM uses kernel=5 to capture 3-candle FVG geometry before recurrence</sup>"),
        "x": 0.5, "xanchor": "center",
    },
    plot_bgcolor="white",
    paper_bgcolor="white",
    height=420,
    margin=dict(t=110, b=30),
    font=dict(size=12),
)

# fig.write_html("../reports/figure15_model_cartoon.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure15_model_cartoon.html")


*Figure 15. XGBoost ingests 35 features that encode domain knowledge about the FVG pattern -- momentum, gap size, session context, Bollinger position. LSTM ingests the raw normalised OHLCV window and must discover all relevant patterns from recurrent state alone. CNN-LSTM adds two convolutional layers before the recurrence: the kernel=5 convolution scans the sequence with a 5-bar receptive field, directly matching the neighborhood of a 3-candle FVG pattern (one bar margin on each side). The hypothesis is that CNN extracts local geometry, LSTM integrates across time -- a better division of labour than raw-input LSTM. The results (CNN-LSTM 0.614 vs LSTM 0.599) partially support this, though neither closes the gap to XGBoost's 35 engineered features.*


In [14]:
"""
Figure 3 — SHAP top features (XGBoost, seed42).
Reads from reports/rigor/2026-05-13/G5/shap_xgb.json.
"""
import json
import plotly.graph_objects as go

with open("../reports/rigor/2026-05-13/G5/shap_xgb.json") as f:
    shap_data = json.load(f)

features = [d["feature"] for d in shap_data["top_5_features"]]
shap_vals = [d["mean_abs_shap"] for d in shap_data["top_5_features"]]

fig = go.Figure(go.Bar(
    x=shap_vals[::-1],
    y=features[::-1],
    orientation="h",
    marker_color="#f59e0b",
    text=[f"{v:.3f}" for v in shap_vals[::-1]],
    textposition="outside",
))
fig.update_layout(
    title={
        "text": (
            "Figure 3. SHAP feature importance — XGBoost seed42 (top 5).<br>"
            "<sup>Mean absolute SHAP value across test set | ValidFVG target</sup>"
        ),
        "x": 0.5, "xanchor": "center"
    },
    xaxis_title="Mean |SHAP|",
    plot_bgcolor="white",
    height=350,
    margin=dict(t=100, l=160, r=80),
    font=dict(size=13),
)
# fig.write_html("../reports/figure03_shap_xgb.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure03_shap_xgb.html")

*Figure 3. The two highest-SHAP features are `ret_60` (60-bar return, capturing trend context) and `gap_norm_bull` (normalised bullish gap size). Both encode geometric properties of the FVG pattern directly. `vol_spike` was dropped from the feature set after this analysis: near-zero SHAP contribution, and removing it gave a negligible +0.001 delta F1.*

---

## 6. What went wrong

MPS GPU deadlock. Multi-layer LSTM training on the MPS device triggered a kernel assertion in `_getLSTMGradKernelDAGObject` during backpropagation. The process hung indefinitely rather than crashing cleanly. Root cause: an unfixed bug in Apple's Metal graph kernel for LSTM backward pass, documented in PyTorch issue #144634 and still open as of August 2025. The fix was to force CPU for all LSTM and CNN-LSTM training. This cost roughly 2x training speed but cleared all hangs (commit `4f64f03`).

Python 3.14 XGBoost subprocess crashes. XGBoost's C extension uses OpenMP for threading. On Python 3.14 (Homebrew), free-threaded ABI changes caused subprocess segfaults when torch had already initialized a thread pool in the parent process. Downgrading to Python 3.12 and running XGBoost in isolated subprocess workers (no torch imports) resolved it (commit `4f64f03`).

Focal loss hypothesis was wrong. The assumption going into the rigor sprint was that focal loss would help with the ~3% positive label. It made every configuration worse. γ=1 dropped mean F1 by 0.064; γ=3 dropped it by 0.109.

Threshold tuning added no value. A PR-curve threshold sweep on the validation set was predicted to recover some minority-class recall. It overfitted the validation distribution instead, hurting test scores for both models.

CNN-LSTM did not beat XGBoost. The hypothesis was that conv kernel=3/5 would capture the 3-candle FVG pattern, giving DL a structural edge. CNN-LSTM (0.614) did beat LSTM (0.599), but both trail XGBoost (0.721) by a wide margin. The 35 hand-engineered features encode more information than the raw OHLCV window at this sample size.

Test-positives assertion bug. An early test asserted at least 200 positive labels in the test split. ValidFVG has only ~163 positives in test. The assertion was written when the raw FVG label was canonical and downgraded to a warning after the migration.

Regularisation checkpoint cache hit. The first run of the G9 regularisation ablation accidentally loaded a cached G2 checkpoint instead of training fresh. The bug was caught by comparing output F1 against the expected range, then fixed by adding a suffix to `_checkpoint_name()` to force separate file paths per ablation config.

---

## 7. What worked

XGBoost is rock-solid. Standard deviation of 0.001 across 5 seeds is essentially zero variation, the model is deterministic at this dataset size, which gives a reliable performance anchor for comparing future architectures.

The ValidFVG labeller validated at κ = 1.0. The 200-sample gold annotation set confirmed perfect agreement between the programmatic labeller and human judgment. This is the most important data quality check in the project, if the labels are wrong, nothing downstream matters.

Pydantic v2 + YAML config system. A full configuration refactor (commits `2947b84`, `c3819ce`) moved all experiment parameters into typed Pydantic schemas backed by YAML files. Adding a new model now requires one Python file and one YAML config, not changes across six scripts. This made the CNN-LSTM sprint noticeably faster to set up.

The subprocess worker for XGBoost works reliably. The workaround for the OpenMP segfault, running XGBoost in an isolated subprocess with no torch import, held across all 5 seeds and all rigor experiments without a single failure.

284/284 tests pass. The test suite covers labels (no-lookahead fixture), models, training utilities, rigor infrastructure, and the live trading harness. All green as of commit `1dfdc4a`.

SHAP confirms the features encode the right pattern. `ret_60` and `gap_norm_bull` dominating the SHAP values is a sanity check: the model is learning from momentum and gap geometry, which is exactly what defines an FVG. It's not finding spurious correlates.

---

## 8. Current state (as of May 13, 2026)

| Item | Status |
|------|--------|
| Data pipeline | Complete, SPY H1 2016-2025, RTH-filtered, labeled, split |
| ValidFVG labeller | Complete, κ=1.0 vs gold annotations |
| XGBoost | Complete, G1-G10 rigor pass, 0.721 ± 0.001 |
| LSTM | Complete, G1-G10 rigor pass, 0.599 ± 0.025 |
| CNN-LSTM | Complete, G1, G2, G4, G6, G7, G9, G10 rigor pass, 0.614 ± 0.021 |
| xLSTM | Not started |
| Transformer | Not started |
| Model inspection tool | Complete, Plotly overlays, trade outcome simulation |
| Paper trading harness | Complete, Alpaca WebSocket, dry-run mode |
| Config system | Complete, Pydantic v2 + YAML-driven |
| Tests | 284/284 passing |

Three models are fully trained and documented. The experimental infrastructure is mature, adding the next architecture requires one config file and one model class. The DL ladder still has two rungs to go.

The current model leaderboard:

```
XGBoost    0.721 ± 0.001    [0.671, 0.761]    ← current best
CNN-LSTM   0.614 ± 0.021    [0.576, 0.649]
LSTM       0.599 ± 0.025    [0.563, 0.635]
Naive      0.328            ,
```

No confidence interval overlap between XGBoost and the DL models. The gap is real, not noise.

In [15]:
"""
Figure 4 — Project timeline: key commits and decisions.
Built from git log. No external data needed.
"""
import plotly.graph_objects as go

# Key events from git log and nb-suite changelogs
events = [
    ("May 8",  "Data pipeline\n(Alpaca pull, resample, split)", "#60a5fa"),
    ("May 8",  "Custom FVG labeller\n(reject smc library, lookahead confirmed)", "#f87171"),
    ("May 10", "ValidFVG labeller\n(6-criteria, κ=1.0 vs gold set)", "#34d399"),
    ("May 11", "LSTM baseline\n(seed42, F1=0.596)", "#a78bfa"),
    ("May 11", "Model inspection tool\n(Plotly overlays, trade outcomes)", "#94a3b8"),
    ("May 11", "Paper trading harness\n(Alpaca WebSocket)", "#94a3b8"),
    ("May 12", "Rigor sprint G1-G10\n(XGB 0.721, LSTM 0.599)", "#f59e0b"),
    ("May 12", "Python 3.12 + CPU stack fix\n(resolve MPS hang + XGB segfault)", "#f87171"),
    ("May 13", "CNN-LSTM sprint\n(0.614 ± 0.021, G1+G2+G4/6/7/9/10)", "#34d399"),
    ("May 13", "Config refactor\n(Pydantic v2 + YAML-driven registry)", "#60a5fa"),
]

x_positions = list(range(len(events)))
labels = [e[0] for e in events]
texts  = [e[1] for e in events]
colors = [e[2] for e in events]

fig = go.Figure()

# Timeline line
fig.add_trace(go.Scatter(
    x=x_positions, y=[0]*len(events),
    mode="lines",
    line=dict(color="#e2e8f0", width=3),
    showlegend=False,
    hoverinfo="skip",
))

# Event dots
fig.add_trace(go.Scatter(
    x=x_positions, y=[0]*len(events),
    mode="markers+text",
    marker=dict(size=18, color=colors, line=dict(width=2, color="white")),
    text=[f"<b>{t}</b><br><span style='font-size:11px'>{d}</span>" for t, d in zip(labels, texts)],
    textposition=["top center" if i % 2 == 0 else "bottom center" for i in range(len(events))],
    hovertext=[t for _, t, _ in events],
    hoverinfo="text",
    showlegend=False,
))

fig.update_layout(
    title={
        "text": "Figure 4. Project timeline — key commits and decisions (May 8–13, 2026).",
        "x": 0.5, "xanchor": "center"
    },
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, range=[-0.5, len(events)-0.5]),
    yaxis=dict(showticklabels=False, showgrid=False, zeroline=False, range=[-2, 3.5]),
    plot_bgcolor="white",
    height=420,
    margin=dict(t=80, b=40, l=20, r=20),
    font=dict(size=12),
)
# fig.write_html("../reports/figure04_timeline.html")  # removed: individual figure exports disabled
fig.show()
# print("Saved: reports/figure04_timeline.html")

*Figure 4. Six days of active development from May 8 to May 13. Red dots mark decisions that required investigation and course correction (library rejection, stack fix). Yellow marks the rigor sprint, the heaviest computation block. All commits are on `master` with conventional commit messages.*

---

## 9. Where we're going

| Deadline | Deliverable | Key work |
|----------|-------------|----------|
| May 17 | Status Update 1 (this notebook) | Peer review |
| June 7 | Status Update 2 | Tutor review, xLSTM + Transformer results expected |
| June 17 | Final presentation + live demo | All models compared, demo on live market data |
| June 20 | Final delivery | Full report, code, results |

### Planned experiments for the next 5 weeks

xLSTM: the extended LSTM architecture from a 2024 paper. Included for academic completeness. Research suggests it may underperform standard LSTM on short financial sequences, but that needs to be verified on the actual data rather than assumed. The config registry already accepts a new model file, so the sprint setup is fast.

Transformer: a lightweight encoder-only architecture using attention over the 60-bar window. Computationally heavier than LSTM but theoretically better at capturing non-adjacent dependencies, for example, a session open 30 bars ago influencing the current bar's FVG quality.

Feature-head DL experiment: give the DL models the same 35 hand-crafted features as XGBoost instead of raw OHLCV. If feature extraction is the bottleneck rather than architecture, this would narrow the gap. Expected lift is +0.05 to +0.10 over raw-OHLCV DL.

Ensemble XGB + CNN-LSTM: average probability outputs from both models. No retraining needed. Expected +0.02 to +0.05 over XGB alone if they make different types of errors.

Live trade smoke test: run the paper trading harness for at least one full trading session with the XGBoost model, confirm the Alpaca WebSocket pipeline works under real market conditions, de-risk the June 17 demo.

The open question for the final report: can DL close the gap on XGBoost once it receives the same feature representation? If yes, architecture matters. If not, the limiting factor is dataset size, and the conclusion shifts toward more data or augmentation as the next direction.

---

## Final Summary

A full ML pipeline to automatically detect high-quality Fair Value Gap zones on SPY hourly charts, built from scratch on a custom-labeled Alpaca dataset.

### Results

| Model | Test Macro F1 | 95% CI |
|-------|--------------|--------|
| XGBoost (5-seed mean) | **0.721 ± 0.001** | [0.671, 0.761] |
| CNN-LSTM (5-seed mean) | 0.614 ± 0.021 | [0.576, 0.649] |
| LSTM (5-seed mean) | 0.599 ± 0.025 | [0.563, 0.635] |
| Naive majority | 0.328 |, |

### Key findings

Classical ML (XGBoost + 35 hand-crafted features) outperforms both DL architectures by 0.107-0.122 macro F1. Non-overlapping bootstrap CIs confirm the gap is statistically real, not sampling noise.

Focal loss is the wrong choice for ~3% positive-rate labels. All γ values degraded F1 relative to WeightedCE, with the worst case (γ=3) dropping F1 by 0.109.

Window size W=60 is well-calibrated. Extending to W=90 gives marginal F1 lift but reduces the independent sample count from 86 to 57, widening uncertainty.

XGBoost SHAP confirms the model learns from the right signals: momentum (`ret_60`) and gap geometry (`gap_norm_bull`) dominate, not noise.

### Limitations

Effective sample size is ~86 independent test windows, bootstrap CIs are wide. More data would narrow uncertainty.

MPS training is disabled due to a PyTorch kernel bug, which capped Optuna trial counts for CNN-LSTM at 6 trials versus 50 for XGB.

ValidFVG criteria thresholds reflect one trader's judgment, validated against one annotator. A multi-annotator study would strengthen the label quality claim.

xLSTM and Transformer remain untested, the DL ladder is not yet complete.

### Next steps

xLSTM sprint, Transformer sprint, feature-head DL experiment, ensemble, live trading smoke test, final report.

---

### References

Joshyattridge (2024). *smart-money-concepts v0.0.27*. GitHub. https://github.com/joshyattridge/smart-money-concepts (accessed May 8, 2026).

PyTorch (2025). *Issue #144634: torch.mps.synchronize hangs on error after kernel assertion*. GitHub. https://github.com/pytorch/pytorch/issues/144634 (accessed May 12, 2026).

PyTorch (2025). *Issue #145374: MPS Memory Leak During LSTM Iterations*. GitHub. https://github.com/pytorch/pytorch/issues/145374 (accessed May 12, 2026).

Akiba, T., Sano, S., Yanase, T., Ohta, T., & Koyama, M. (2019). Optuna: A Next-generation Hyperparameter Optimization Framework. *Proceedings of the 25th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*. https://doi.org/10.1145/3292500.3330701

Lundberg, S. M., & Lee, S. (2017). A Unified Approach to Interpreting Model Predictions. *Advances in Neural Information Processing Systems, 30*. https://proceedings.neurips.cc/paper/2017/hash/8a20a8621978632d76c43dfd28b67767-Abstract.html

Lin, T., Goyal, P., Girshick, R., He, K., & Dollar, P. (2017). Focal Loss for Dense Object Detection. *IEEE International Conference on Computer Vision (ICCV)*. https://doi.org/10.1109/ICCV.2017.324

---

*LO evidence: LO1, data provenance (Section 3), split integrity, lookahead prevention via N+1 label index enforced by pytest fixture. LO2, model comparison from baseline (Section 4), metric justification (macro F1 over accuracy on imbalanced data), ablation table (Section 5), limitations stated. LO3, CRISP-DM methodology structures this notebook; every major decision documented with reason (library rejection, loss function, threshold tuning). LO4, documented course corrections (MPS fix, Python downgrade, label migration), honest failure accounting (Section 6).*